<a href="https://colab.research.google.com/github/kiwi8803/Biometric/blob/main/Assign3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CS5332 Assignment 3** </br>
**Name: $Lim\ Kian\ Hwee$** </br>
**Student number: A$0329838L$**



### Using Google Colab, upload this notebook and the codebase to your Google Drive, under CS5332_Assign3. Then execute the next 3 cells to mount your Google Drive in Colab and set your working directory.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
root_dir = "/content/drive/My Drive/"
project_dir = "CS5332_Assign3/" # Change to your path
os.chdir(root_dir + project_dir)

In [4]:
# Make sure the path is correct
!ls

Assign3.ipynb  Assignment3.pdf	features  gait	predict_gender_stat.py	utils


In [5]:
## Example code: this trains a gender classifier
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# number of IDs in the dataset
nID = 40

# gender labels, taken from demographics.csv
# you need to change this to the apropriate label/value for each task
labels = [0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1]

# path to the feature set
# change this path to the folder which contains the statistical features which were given
currPath = ("features/")

# read in the stat features
X = np.genfromtxt(currPath + "30.csv", delimiter=',', skip_header=False)
# read in the IDs of each feature vector
input_ids = np.genfromtxt(currPath + "30ids.csv", delimiter=',', skip_header=False)

# prepare y with appropriate label for each training feature
y = np.zeros(input_ids.shape)
for idx, input_id in enumerate(input_ids):
    y[idx] = labels[int(input_id) - 1]

# split X and y into training and testing sets
# all ids until 30 are used for training
# all ids from 31 to 40 are used for testing
train_x = X[np.where(input_ids[:] <= 30)]
train_y = y[np.where(input_ids[:] <= 30)]

test_x = X[np.where(input_ids[:] > 30)]
test_y = y[np.where(input_ids[:] > 30)]
test_ids = input_ids[np.where(input_ids[:] > 30)]

# train classifier for gender prediction
# you need to change this for regression tasks
model = RandomForestClassifier(max_depth=25, n_estimators=50, max_features=1)
model.fit(train_x, train_y)

# evaluation
# you need to change the evaluation for each task
pred = model.predict(test_x)
score = model.score(test_x, test_y)

print("Per window accuracy:", score)

correctCounts = np.zeros((nID+1,))
totalCounts = np.zeros((nID+1,))
for i in range(0, test_y.__len__()):
    currID = int(test_ids[i])
    currPred = pred[i]
    currActual = test_y[i]
    if (currPred == currActual):
        correctCounts[currID] += 1
    totalCounts[currID] += 1

correct = 0

# loop through testing samples
# use threshold = 0.5, to give classification for each class
for k in range(31, 41):
    if correctCounts[k] / totalCounts[k] >= 0.5:
        correct += 1

print("Per walk accuracy", correct / 10)


Per window accuracy: 0.6327959465684017
Per walk accuracy 0.8


In [13]:
from ast import Param
## Task 5.1: insert your code here
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


# Feature scaling for transformation to be learned only from trained data (ID :1-30)

rf_pipeline = Pipeline([
    ('rf', RandomForestClassifier(n_estimators = 200, max_depth = 25, max_features = 'sqrt', random_state=np.random.RandomState(42)))
])
# Tuning
#param_grid_rf = {
   # 'rf__n_estimators': [50,100,200],
   # 'rf__max_depth': [10,20, 25 ,50, None ],
    ##'rf__max_features': ['sqrt', 'log2' , 1] # Moving beyond the baseline of 1
#}

# 3. Use GridSearchCV for the Training Phase (IDs 1-30)
# This will perform 5-fold Cross-Validation [15]
#grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=5, n_jobs=-1)
rf_pipeline.fit(train_x, train_y)
rf_preds = rf_pipeline.predict(test_x)
rf_score = rf_pipeline.score(test_x, test_y)
#print(f"Best Parameters for Gender: {rf_pipeline.best_params_}")

# 2. Re-use your Per-Walk Evaluation Logic
def get_per_walk_accuracy(predictions, test_ids, labels):
    correct_walks = 0
    for k in range(31, 41):
        idx = np.where(test_ids == k)
        if len(idx) == 0: continue

        # Majority vote (threshold 0.5) [11, 12]
        person_preds = predictions[idx]
        final_vote = 1 if np.mean(person_preds) >= 0.5 else 0

        if final_vote == labels[k-1]:
            correct_walks += 1

    return correct_walks / 10

print("Per window accuracy:", rf_score)
print(f"Per-Walk Accuracy: {get_per_walk_accuracy(rf_preds, test_ids, labels)}")


Per window accuracy: 0.6353754030400737
KNN Per-Walk Accuracy: 0.8


In [10]:
## Task 5.2: insert your code here
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet

import numpy as np

# number of IDs in the dataset
nID = 40

# Prepare Demographics Data(Ground Truth)
height_cm = [152,180,173,170,172,156,170,176,176,170,154,168,157,173,164,164,167,160,172,152,172,176,173,146,160,173,168,195,167,152,163,160,170,169,161,153,160,172,158,172]
weight_kg = [52.3,83.8,69.2,68.2,66.8,66.1,60.5,73,68.7,65,53,71,78.7,77,73.7,71.6,76,60.5,60.5,46.6,76.8,68.1,71.6,53.1,54.3,67.2,78.8,95.6,66.9,57.7,83,52.4,61.3,69.6,54,51,55.1,75.8,68.9,74.9]
# Define pipeline

# path to the feature set
# change this path to the folder which contains the statistical features which were given
currPath = ("features/")

# read in the stat features
X = np.genfromtxt(currPath + "30.csv", delimiter=',', skip_header=False)
# read in the IDs of each feature vector
input_ids = np.genfromtxt(currPath + "30ids.csv", delimiter=',', skip_header=False)

# map window IDs to continous Labels (weight and height)
y_weight = np.array([weight_kg[int(i) -1] for i in input_ids])
y_height = np.array([height_cm[int(i) -1] for i in input_ids])

train_x = X[np.where(input_ids[:] <= 30)]
# Split into train (IDs 1-30) and test (IDs 31-40)
train_y_w = y_weight[np.where(input_ids <= 30)]
train_y_h = y_height[np.where(input_ids <= 30)]

test_x = X[np.where(input_ids[:] > 30)]
test_y_height = y_height[np.where(input_ids > 30)]
test_y_weight = y_weight[np.where(input_ids > 30)]
test_ids = input_ids[np.where(input_ids > 30)]

en_pipeline = Pipeline([
    ('scaler', StandardScaler()), # Mandatory for linear models [13]
    ('pca', PCA(n_components=0.95, whiten=True)), # Increased energy threshold [21]
    ('regr', ElasticNet(random_state=42, alpha =10, l1_ratio = 0.5))
])

# 2. Tune Alpha (strength) and l1_ratio (balance between Ridge and Lasso)
#param_grid_en = {
#    'regr__alpha': [0.01, 0.1, 1, 10],
#    'regr__l1_ratio': [0.1, 0.5, 0.9] # Handles correlated features [12]
#}
#grid_en = GridSearchCV(improvement_pipeline, param_grid_en, cv=5, scoring='neg_mean_squared_error')
#grid_en.fit(train_x, train_y_w) # Train on IDs 1-30
#print(f"Best parameters for weight prediction: {grid_en.best_params_}")
## RANDOMFORESTREGRESSOR
reg_pipeline = Pipeline([
    ('regr', RandomForestRegressor(n_estimators=50, random_state=42))
])
##tuning...
#param_grid_reg = {
#    'regr__n_estimators': [50, 100, 200],       # Number of trees
#    'regr__max_depth': [3, 5, 10, None],        # Keep the trees shallow!
#    'regr__min_samples_split': [2, 5, 10],      # Force trees to generalize
#    'regr__min_samples_leaf': [1, 2, 4]         # Prevent single-sample leaves
#}
def evaluate_per_walk_rmse(model, test_x, test_ids, ground_truth, is_height=False):
    preds = model.predict(test_x)
    per_person_avg_preds = []
    actual_values = []

    for k in range(31, 41):
        idx = np.where(test_ids == k)
        if len(idx) == 0: continue

        # Average per-window predictions for each person [2]
        avg_prediction = np.mean(preds[idx])

        # Convert height from cm to m if necessary [3, 4]
        if is_height:
            per_person_avg_preds.append(avg_prediction / 100.0)
            actual_values.append(ground_truth[k-1] / 100.0)
        else:
            per_person_avg_preds.append(avg_prediction)
            actual_values.append(ground_truth[k-1])

    rmse = np.sqrt(mean_squared_error(actual_values, per_person_avg_preds))
    return rmse

# Train and Evaluate
# Weight Prediction
reg_pipeline.fit(train_x, train_y_w)
#grid_en.fit(train_x, train_y_w)
weight_rmse = evaluate_per_walk_rmse(reg_pipeline, test_x, test_ids, weight_kg)
print(f"Weight Prediction RMSE: {weight_rmse:.4f} kg")
height_m = np.array(height_cm) /100.0
#find average error
avg_height = np.mean(height_m[:30])
avg_weight = np.mean(weight_kg[:30])

#print relative RMSE (percentage error)
print(f"Relative Height Error: {(weight_rmse/avg_weight)*100:.4f} %")
# Height Prediction
#reg_pipeline.fit(train_x, train_y_h)
en_pipeline.fit(train_x, train_y_h)
height_rmse = evaluate_per_walk_rmse(en_pipeline, test_x, test_ids, height_cm, is_height=True)
print(f"Height Prediction RMSE: {height_rmse:.4f} m")
print(f"Relative Height Error: {(height_rmse/avg_height)*100:.4f} %")






Weight Prediction RMSE: 10.5765 kg
Relative Height Error: 15.6126 %
Height Prediction RMSE: 0.0674 m
Relative Height Error: 4.0363 %


In [9]:
## Task 5.3: Calucualte the BMI for each person using their actual height and weight values

#Ground Truth list
#convert M to cm
height_m = np.array(height_cm) /100.0

#BMI calculation
actual_bmi_list = np.divide(weight_kg, np.square(height_m))
print(actual_bmi_list)


[22.63677285 25.86419753 23.12138728 23.59861592 22.57977285 27.16140697
 20.93425606 23.56663223 22.17846074 22.49134948 22.34778209 25.15589569
 31.92827295 25.72755521 27.40184414 26.62105889 27.25088745 23.6328125
 20.45024337 20.16966759 25.95997837 21.9847624  23.92328511 24.91086508
 21.2109375  22.4531391  27.91950113 25.14135437 23.98795224 24.97403047
 31.23941436 20.46875    21.21107266 24.36889465 20.83252961 21.78649237
 21.5234375  25.62195782 27.59974363 25.31773932]


In [23]:
## Task 5.4: Use BMI Value calculated above to train a regressor to predict the BMI Value

#map actual BMI to every gait window
y_bmi = np.array([actual_bmi_list[int(i) - 1] for i in input_ids])
#slipt into Train and Test
train_y_bmi = y_bmi[np.where(input_ids <= 30)]
test_y_bmi = y_bmi[np.where(input_ids > 30)]

avg_bmi = np.mean(actual_bmi_list[:30])
# Train the Direct Regressor
# Using the same pipeline structure as Task 2
bmi_direct_pipeline = Pipeline([
    ('regr', RandomForestRegressor(n_estimators=50, max_depth=25, random_state=42))
])
bmi_direct_pipeline.fit(train_x, train_y_bmi)
y_pred = bmi_direct_pipeline.predict(test_x)
y_score = bmi_direct_pipeline.score(test_x, test_y_bmi)
print("Per window accuracy:", y_score)

# Average per walk/person
def print_per_walk_results(predictions, test_ids, actual_bmi_list):
    for k in range(31, 41):
        idx = np.where(test_ids == k)[0]
        if len(idx) == 0:
            continue

        pred_mean = np.mean(predictions[idx])
        actual = actual_bmi_list[k - 1]

        print(f"Person {k} -> Predicted: {pred_mean:.2f}, Actual: {actual:.2f}")

print_per_walk_results(y_pred, test_ids, actual_bmi_list)

# 4Evaluate using Per-Walk RMSE
# (Using the averaging function defined in Task 2)
direct_bmi_rmse = evaluate_per_walk_rmse(bmi_direct_pipeline, test_x, test_ids, actual_bmi_list)
print(f"Direct BMI Prediction RMSE: {direct_bmi_rmse:.4f} kg·m^-2")
print(f"Relative BMI Error: {(direct_bmi_rmse/avg_bmi)*100:.4f} %")
# find average error

Per window accuracy: -0.19313690131894878
Person 31 -> Predicted: 23.87, Actual: 31.24
Person 32 -> Predicted: 24.32, Actual: 20.47
Person 33 -> Predicted: 24.59, Actual: 21.21
Person 34 -> Predicted: 25.21, Actual: 24.37
Person 35 -> Predicted: 24.63, Actual: 20.83
Person 36 -> Predicted: 23.21, Actual: 21.79
Person 37 -> Predicted: 24.48, Actual: 21.52
Person 38 -> Predicted: 24.49, Actual: 25.62
Person 39 -> Predicted: 24.41, Actual: 27.60
Person 40 -> Predicted: 24.57, Actual: 25.32
Direct BMI Prediction RMSE: 3.4422 kg·m^-2
Relative BMI Error: 14.1987 %


In [28]:
## Task 5.5: Calculate indirect bmi rsme

def calculate_indirect_bmi_rmse(weight_model, height_model, test_x, test_ids, ground_truth):
  #get perwindow predict
  w_preds = weight_model.predict(test_x)
  h_preds = height_model.predict(test_x)

  indirect_bmi_avgs = []
  actual_values = []

  for k in range(31, 41):
    idx = np.where(test_ids == k)
    if len(idx) == 0: continue

    #average predeicted weight and height for this subject
    avg_weight = np.mean(w_preds[idx])
    avg_height = np.mean(h_preds[idx]) /100 # convert cm to m

    # calculate BMI for average prediction
    pred_bmi = avg_weight / np.square(avg_height)
    indirect_bmi_avgs.append(pred_bmi)
    #actual BMI for comparison
    actual_values.append(ground_truth[k-1])
    print(f"Person {k} -> Predicted BMI: {pred_bmi:.2f}, Actual BMI: {actual_bmi_list[k-1]:.4f}")
    # calculate BMI
  rmse = np.sqrt(mean_squared_error(actual_values, indirect_bmi_avgs))
  return rmse

indirect_bmi_rmse = calculate_indirect_bmi_rmse(reg_pipeline, en_pipeline, test_x, test_ids, actual_bmi_list)
print(f"Indirect BMI Prediction RMSE: {indirect_bmi_rmse:.4f} kg·m^-2")
print(f"Relative BMI Error: {(indirect_bmi_rmse/avg_bmi)*100:.4f} %")

Person 31 -> Predicted BMI: 24.26, Actual BMI: 31.2394
Person 32 -> Predicted BMI: 24.46, Actual BMI: 20.4687
Person 33 -> Predicted BMI: 24.94, Actual BMI: 21.2111
Person 34 -> Predicted BMI: 24.89, Actual BMI: 24.3689
Person 35 -> Predicted BMI: 23.85, Actual BMI: 20.8325
Person 36 -> Predicted BMI: 23.20, Actual BMI: 21.7865
Person 37 -> Predicted BMI: 23.49, Actual BMI: 21.5234
Person 38 -> Predicted BMI: 24.19, Actual BMI: 25.6220
Person 39 -> Predicted BMI: 24.41, Actual BMI: 27.5997
Person 40 -> Predicted BMI: 23.93, Actual BMI: 25.3177
Indirect BMI Prediction RMSE: 3.2854 kg·m^-2
Relative BMI Error: 13.5519 %


**Answer for Q1:**
The Accuracy of Task 1 is 80% per walk and overall per window of 64.2%, The model used is RandomForestClassifier. The parameter used is as follows 'max_depth': 25, 'max_features': 'log2', 'n_estimators': 200. These paramemters was derived using tuning

This was after comparing between KNN, SVC and Naive Bayes and results was observed that RandomForestClassifer achieved higher accuracy. for example KNN's Per window accuracy: 58.5%.

This might be due to the higher dimensional gait data on the data and features might have vastly difference ranges and scale. for Naive Bayes, it assumes conditional independence.

Task2's uses the combination

* Height Prediction RMSE: 0.0674 m, ElasticNet(random_state=42, alpha =10, l1_ratio = 0.5) chosen to capture linear trends
* Weight Prediction RMSE: 10.5765 kg, RandomForestRegressor(n_estimators=50, random_state=42)

**Answer for Q2:**
Task4's direct BMI prediction, uses a RandomForestRegressor with the following parameters n_estmators = 50, maxdepth = 25m random = 42.  

Task5's indirect BMI prediction, uses the combination
* for Height, ElasticNet(random_state=42, alpha =10, l1_ratio = 0.5)
* for Weight, RandomForestRegressor(n_estimators=50, random_state=42)

Task 4's RMSE: 3.4422 kg·m^-2.  
Task 5's RMSE: 3.2854 kg·m^-2.
---
| Person | Direct BMI | Indirect BMI | Actual BMI |
|--------|------------|--------------|------------|
| 31     |23.87(-7.73)| 24.26(-6.98) | 31.24      |
| 32     |24.32(+3.75)| 24.46(+3.89)| 20.57      |
| 33     |24.59(+3.38)| 24.94(+3.73)| 21.21      |
| 34     |25.21(+0.83)| 24.89(+0.52)| 24.37      |
| 35     |24.63(+3.80)| 23.85(+3.02)| 20.83      |
| 36     |23.21(+1.42)| 23.20(+1.41)| 21.79      |
| 37     |24.48(+2.92)| 23.49(+1.97)| 21.52      |
| 38     |24.49(-1.13)| 24.19(-1.43)| 25.62      |
| 39     |24.41(-3.19)| 24.41(-3.19)| 27.60      |
| 40     |24.57(-0.75)| 23.93(-1.39)| 25.32      |



---

  Indirect BMI prediction outperforms direct BMI prediction. This is likely due to the BMI depends on both height and weight which are independent qualities. The direct BMI prediction then tries to infer a nonlinear combination of height^2 and weight simultaneously. also for people with unsual BMI(very high or low), the model seems to regress towards the mean. Whereas for Indirect BMI prediction, the model could predict it from gait at a low relative error (~4%) stablising the BMI calculation hence lowering the RMSE.

**Answer for Q3:**
Based on the result, **height** could be predicted well using gait signal. we compare the normalised errors of each prediction to Relative Error of RMSE/average :
- Height RMSE: 0.0674 m - Relative Error 4.0363 %
- Weight RMSE: 10.5765 kg - Relative Error 15.6126 %
- BMI  RMSE: 3.4422 kg·m^-2  - Relative Error 14.1987  %

**Answer for Q4:**
Gait derived attributes such as height, weight, and BMI can be applied to several real world application.
- Security and Surveillance, these attributes can enhance personnel tracking and re-identification across non-overlapping cameras, or occulsion, making such system more robust to occulsion or poort visibility.

- Health Monitoring. In controlled environment such as the military bases, it can support health monitoring by identify personnel at risk of high BMI and enable intervention and tracking for fitness intervention

- Targeted advertisement, where smart system could tailor digital content based on inferred body type.

**Answer for Q5:**


Gait-derived BMI constitutes latent health data, enabling probabilistic inference of conditions such as obesity-related risks, thereby transforming seemingly innocuous behavioural data into sensitive medical indicators. Unlike traditional biometrics, gait can be passively captured at a distance using existing surveillance infrastructure, eliminating the need for user interaction or awareness. This enables covert tracking and profiling of individuals, creating opportunities to exploit vulnerabilities for monetary gain (e.g., targeted advertising) or discriminatory practices (e.g., automated enforcement of stricter training regimes in military contexts).Gait data exists at the intersection of territorial privacy (tracking movement in public spaces) and bodily privacy (inferring biomechanical and physiological traits), thereby amplifying its sensitivity and potential for misuse.

The recommendation to protect privacy as follows:
- First, the deployment of gait-based biometric inference should be restricted to high-risk domains such as critical infrastructure security, where its use can be justified under the principle of proportionality. Second, organisations must ensure transparency and consent, particularly in relation to bodily privacy. Clear signage and policy disclosures should inform individuals of what data is being collected and what inferences are derived, enabling informed opt-in consent for non-security applications such as healthcare support.

- From a policy perspective, such data should be protected under the Personal Data Protection Act (Singapore). However, while the PDPA provides a foundational framework, it struggles to adequately address inferred data. While raw gait video clearly constitutes personal data, gait-derived BMI is not explicitly defined, despite revealing highly sensitive information. Such inferred attributes should be treated as health-related data from the point of derivation and afforded a higher level of protection due to their potential for harm.

- To address this gap, regulation should enforce strict purpose limitation, ensuring that systems developed for one purpose cannot be repurposed for another without conducting a new Data Protection Impact Assessment (DPIA) and providing public notification. This would reduce the risk of function creep and strengthen accountability in the use of biometric inference systems.

